# GRPO on MBPP — local (RTX 4070) — headroom experiment

RL post-training with verifiable code-execution rewards.

**This run uses the *general* Qwen2.5-1.5B-Instruct** (not the Coder variant) so MBPP isn't saturated and there's room for the reward to transfer. Keep your earlier Coder result (0.62 → 0.62 null) for the comparison.

**4070 notes:** Ada supports bf16, so the config auto-uses it (more stable than fp16). vLLM is left off — generation goes through HF `generate`, which avoids the whole FlashInfer/CUDA-stub mess.

**Run order:** fresh kernel, top to bottom. Cell 5 (baseline) must run *before* cell 6 (train).

In [ ]:
# 0) Install (skip if your env already has these). Restart kernel after if numpy warns.
# %pip install -q unsloth
# In a clean local venv with a recent torch+CUDA, `pip install unsloth` pulls
# trl / peft / transformers / bitsandbytes / datasets at compatible versions.

In [ ]:
# 1) Load model (4-bit) + LoRA  —  vLLM OFF (HF generate path)
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"            # general base — HAS headroom (this run)
# MODEL_NAME = "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit"   # code base — saturated (the 0.62 null)

max_seq_length, lora_rank = 2048, 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = max_seq_length,
    load_in_4bit   = True,
    fast_inference = False,        # no vLLM/FlashInfer
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
model.print_trainable_parameters()
print("bf16 supported:", torch.cuda.is_bf16_supported(), "| device:", next(model.parameters()).device)

In [ ]:
# 2) MBPP -> chat prompts  (needs internet for first download)
from datasets import load_dataset

SYSTEM = ("You are an expert Python programmer. Write a single self-contained "
          "Python function that solves the problem. Return only the function "
          "inside a ```python code block.")

def make_example(ex):
    tests = "\n".join(ex["test_list"])
    user  = f"{ex['text']}\n\nYour function must pass these tests:\n{tests}"
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM},
            {"role": "user",   "content": user},
        ],
        "test_list":       ex["test_list"],
        "test_setup_code": ex.get("test_setup_code", "") or "",
    }

raw = load_dataset("mbpp", split="train")            # 374 problems
ds  = raw.map(make_example, remove_columns=raw.column_names)
print(ds)
print(ds[0]["prompt"][-1]["content"][:300])

In [ ]:
# 3) Reward functions: subprocess-isolated executor + format nudge
import os, re, json, tempfile, subprocess, sys, resource

HARNESS = r"""
import sys, json
with open("meta.json") as f: meta = json.load(f)
with open("solution.py") as f: src = f.read()
ns = {}
try:
    if meta.get("setup"): exec(meta["setup"], ns)
    exec(src, ns)
except Exception:
    print("__PASSED__0__"); sys.exit(0)
passed = 0
for t in meta["tests"]:
    try:
        exec(t, ns); passed += 1
    except Exception:
        pass
print(f"__PASSED__{passed}__")
"""

def extract_code(text):
    m = re.search(r"```(?:python)?\s*\n(.*?)```", text, re.DOTALL)
    return m.group(1) if m else text

def _limits(timeout):
    def _set():
        resource.setrlimit(resource.RLIMIT_CPU, (timeout, timeout))
        resource.setrlimit(resource.RLIMIT_AS, (2 * 1024**3, 2 * 1024**3))
    return _set

def run_tests(code_str, setup_str, test_list, timeout=8):
    n = len(test_list)
    if n == 0: return 0.0
    with tempfile.TemporaryDirectory() as d:
        open(os.path.join(d, "solution.py"), "w").write(code_str)
        json.dump({"setup": setup_str, "tests": test_list},
                  open(os.path.join(d, "meta.json"), "w"))
        open(os.path.join(d, "harness.py"), "w").write(HARNESS)
        try:
            out = subprocess.run([sys.executable, "harness.py"], cwd=d,
                                 capture_output=True, text=True,
                                 timeout=timeout, preexec_fn=_limits(timeout))
        except Exception:
            return 0.0
    m = re.search(r"__PASSED__(\d+)__", out.stdout)
    return int(m.group(1)) / n if m else 0.0

def _completion_text(c):
    if isinstance(c, str): return c
    if isinstance(c, list) and c and isinstance(c[-1], dict):
        return c[-1].get("content", "")
    return str(c)

def code_reward(completions, test_list, test_setup_code, **kwargs):
    return [run_tests(extract_code(_completion_text(c)), s or "", t, timeout=8)
            for c, t, s in zip(completions, test_list, test_setup_code)]

def format_reward(completions, **kwargs):
    out = []
    for c in completions:
        text = _completion_text(c)
        out.append(0.2 if re.search(r"```(?:python)?\s*\n.*?```", text, re.DOTALL) else 0.0)
    return out

In [ ]:
# 4) Sanity-check the executor (must print 1.0 / 0.333 / 0.0)
tests = ["assert add(2,3)==5", "assert add(0,0)==0", "assert add(-1,1)==0"]
print(run_tests("def add(a,b): return a+b", "", tests))
print(run_tests("def add(a,b): return a-b", "", tests))
print(run_tests("def add(a,b):\n while True: pass", "", tests))

In [ ]:
# 5) Held-out pass@1 baseline. RUN BEFORE TRAINING.
from datasets import load_dataset

test_raw = load_dataset("mbpp", split="test").select(range(50))   # unseen split

def evaluate(model, tokenizer, examples):
    FastLanguageModel.for_inference(model)
    solved = 0
    for ex in examples:
        msgs = [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"{ex['text']}\n\nYour function must pass these tests:\n" + "\n".join(ex["test_list"])},
        ]
        ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
        out = model.generate(input_ids=ids, max_new_tokens=512, do_sample=False)
        text = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
        frac = run_tests(extract_code(text), ex.get("test_setup_code","") or "", ex["test_list"])
        solved += (frac == 1.0)
    return solved / len(examples)

baseline = evaluate(model, tokenizer, test_raw)
print("BEFORE GRPO  pass@1:", baseline)

In [ ]:
# 5b) OPTIONAL — train only on problems the base currently FAILS (more headroom per step).
# Leave OFF for your first general-base run (keeps it a clean one-variable change vs the Coder run).
USE_FAILED_FILTER = False

if USE_FAILED_FILTER:
    FastLanguageModel.for_inference(model)
    def base_fails(ex):
        msgs = [{"role":"system","content":SYSTEM},
                {"role":"user","content": f"{ex['text']}\n\nYour function must pass these tests:\n" + "\n".join(ex["test_list"])}]
        ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
        out = model.generate(input_ids=ids, max_new_tokens=512, do_sample=False)
        text = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
        return run_tests(extract_code(text), ex.get("test_setup_code","") or "", ex["test_list"]) < 1.0
    keep = [i for i, ex in enumerate(raw) if base_fails(ex)]
    print(f"kept {len(keep)}/{len(raw)} currently-failed problems")
    ds = ds.select(keep)

In [ ]:
# 6) GRPO training. bf16 auto on Ada (4070). Smoke test = 20; bump to 150 for the real run.
import torch
from trl import GRPOConfig, GRPOTrainer

use_bf16 = torch.cuda.is_bf16_supported()   # True on 4070, False on T4

cfg = GRPOConfig(
    output_dir = "grpo_qwen_local",
    use_vllm   = False,
    learning_rate = 5e-6, optim = "adamw_8bit",
    lr_scheduler_type = "cosine", warmup_ratio = 0.1,
    weight_decay = 0.1, max_grad_norm = 0.1,
    bf16 = use_bf16, fp16 = not use_bf16,
    per_device_train_batch_size = 4,     # must be divisible by num_generations; raise to 8 if VRAM allows
    gradient_accumulation_steps = 1,
    num_generations = 4,
    max_prompt_length = 512,
    max_completion_length = 512,         # drop to 384 if you OOM
    max_steps = 20,                      # <-- bump to 150 once smoke test looks good
    logging_steps = 1, save_steps = 100, report_to = "none",
)

FastLanguageModel.for_training(model)
trainer = GRPOTrainer(
    model = model, processing_class = tokenizer,
    reward_funcs = [code_reward, format_reward],
    args = cfg, train_dataset = ds,
)
trainer.train()

In [ ]:
# 7) After-eval + delta. Same held-out split. This is your result.
after = evaluate(model, tokenizer, test_raw)
print("BEFORE GRPO  pass@1:", baseline)
print("AFTER  GRPO  pass@1:", after)
print("delta       :", round(after - baseline, 4))

In [ ]:
# 8) Save the LoRA adapter
model.save_pretrained("grpo_qwen_local_lora")
tokenizer.save_pretrained("grpo_qwen_local_lora")
print("saved -> grpo_qwen_local_lora/")